# 9주차 실습 — 분석 질문을 SQL로 바꾸기

- **데이터**: `aact_mini` — 임상시험 37,509건(2015~2024년 시작). 7 · 8주차와 같은 데이터입니다.
- **오늘 배우는 것**: ① 질문을 다섯 단계로 나누어 SQL로 바꾸기 · ② 서브쿼리로 대상을 먼저 뽑아 세기 · ③ 조건을 하나씩 더하며 확인하기 · ④ 검산 다섯 가지 · ⑤ 과제 1 제출
- 이 실습은 점수 없음. 과제 1 (20점) — 7 · 8주차 범위, 9주차 수업 후 1주일 이내 LMS 제출

## 0. 시작하기 전에

- Colab 에서는 먼저 **`파일 → Drive에 사본 저장`** 을 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` 같은 줄은 지우지 마세요.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "aact_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/aact_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 50MB, 1분 내외) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 학번 셀과 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "바로 위 SQL 셀에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 SQL을 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, f"결과가 표가 아닙니다. %%sql 셀을 실행한 뒤, 바로 다음 셀에서 check('{name}', _) 를 실행하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 check 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

In [ ]:
#@title [2번] 자동 확인 도우미 — 정답 SQL 은 들어 있지 않습니다 (수정하지 마세요) { display-mode: "form" }
ANSWERS = _load_answers("eNqtlF9P2mAUxr/Km17JQpa2UDt2t2zXLibb1bIYCmUzIbIAbhfGBLQuKCyKgJatJZAwRW/WzTo18dN42XP6HXbeVmIhuoxkV+379Jw3fX7nz5qwKAlP2ZpQWn5HTyGZUxO6IqY0OZUR4kwoPKAXC59KpEv0mink+esbAeuWf2hizRTekvx+eaXMMz3X8M1Lhp0aHu6hYbGbigmNCm5WWZjA4FeNR8DpCQzsm0oXbYPBcQNaQ++sH2fgtL2r72Ey7LSZdzGOvb2UQSv4EF5HmYzy4GgL6jWoDx8HNopZvahn6X9y6XxJ58qH8nJhJZ0fS+ukLcpRFKKuKqKcS+iamI2imNbvQxEa5F+hPvLOjCXs1dBuTIB5/vL1wqu5RzHG/x1OR9DqM/9gm9zHWfgND90YQ7sSWKYD9+o5+2i7jO7jaRTP/eLW9uxmE1GzGUVNZKSULOmKFjU7rf+97txvo+JdWGRn4rAEVwZaown/1AeKKMLpFll0iRZVPjAbirBLTdAa4oHrOc1bc2w+pMnQrGHfQttkOHC8n653dk1gurhzPjuG5ETNkxk5IanifCaZnKj5lP4fMcCGCfY1ZwBHBvcPXxw6+IbDy439qt+jAptVInEbxNuBroSBFac+aFIOtB00Xb9pjScC29WQBJubYtatBKR4Uzn7sVlAKVFQWU0SdVlT5JSkREFN6/++J7DTmywoH4kmDfdxDzsGdq6Z3zHhB7fNni28YPzpd8jx0OALg/cQEYLfXeyNJ59RV/mbVjA6ZLjwUS+m8/mlUjldXi2xu7gZIMxHIaTUVFbVxSeqpklRCNP6vRDOLfzcQPsSv+3xGDrzpRGeo1wWJca53K3RObQIB62Mh5dobDwQvJ02XPx6EiDqV2eYkHJxVV9f/wMk1rWG")

## 1. 질문을 SQL로 바꾸기 — 다섯 단계 절차

### 분석 질문을 SQL로 바꾸는 절차 — 다섯 단계를 순서대로

| 단계 | 할 일 |
|---|---|
| ① 질문을 한 문장으로 | “무엇을, 어떤 대상에서, 몇 개” 를 분명히 적는다 |
| ② 세는 단위를 정한다 | 시험 건수? 환자 수? 기관 수? — 결과의 한 행이 무엇인가 |
| ③ 필요한 표와 연결 키 | 어느 표에 있는 값인가, 무엇으로 연결하는가 |
| ④ 조건을 하나씩 더한다 | 한 번에 다 쓰지 않고 건수 변화를 보며 추가 |
| ⑤ 검산한다 | 건수 · 분모 · 중복을 확인한 뒤 결과를 쓴다 |

SQL을 먼저 쓰지 말고, ①②③을 **말이나 글로 먼저** 적는 것이 가장 빠릅니다.

### 같은 질문, 다른 답 — 세는 단위를 정하지 않으면 생기는 일

“당뇨 임상시험이 몇 건인가요?” — 이 질문만으로는 SQL을 쓸 수 없습니다.

| | 이렇게 물으면 — 모호함 | 이렇게 물으면 — 분명함 |
|---|---|---|
| 질문 | 당뇨 시험이 몇 건인가요? | 2015~2024년에 시작한, MeSH 용어가 'Diabetes Mellitus, Type 2' 인, 중재 연구 중 완료된 것이 몇 **건**인가요? |
| 결과 | 제2형만? 제1형도? 진행 중인 것만? 완료된 것도? 어느 나라? 어느 기간? → 사람마다 다른 숫자가 나옴 | 누가 해도 같은 숫자 |

분석을 요청받으면 먼저 네 가지를 되묻습니다: **대상 · 기간 · 조건 · 세는 단위**.

11주차 요청서에서 다시 봅니다.

### 서브쿼리 — 대상을 먼저 뽑고, 그 안에서 세기

괄호 안이 먼저 계산됩니다.

괄호 안에서 ‘대상자 목록’을 먼저 만들고, 바깥에서 그 목록에 드는 것만 셉니다.

In [ ]:
%%sql
-- [3번] “제2형 당뇨 대상 시험이 몇 건이야?” → 1,353
SELECT COUNT(DISTINCT nct_id) AS 시험수
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Diabetes Mellitus, Type 2'
);

### 서브쿼리와 JOIN — 언제 무엇을

같은 답이 나오지만 쓰임이 다릅니다.

| | 서브쿼리 (IN) | JOIN |
|---|---|---|
| 무엇을 하나 | 대상 목록을 만들어 걸러냄 | 두 표를 옆으로 이어 붙임 |
| 행 수가 늘어나나 | 늘지 않음 (원래 표 그대로) | 1 : N 이면 늘어남 |
| 언제 쓰나 | ‘이 조건에 맞는 대상만’ 고를 때 | 다른 표의 **열을 함께 보여줄** 때 |
| 이번 사례에서는 | 대상 시험 고르기 | 상태 · 국가를 함께 보기 |

대상을 고르기만 할 때는 서브쿼리가 안전합니다. **행 수가 변하지 않기** 때문입니다.

## 2. 종합 예제 — 데이터 파악부터 보고까지

제2형 당뇨병 시험으로 분석 순서 0~7단계를 따라 갑니다.

| 준비 | 정리 | 분석 | 보고 |
|---|---|---|---|
| 0 파악 · 1 대상 · 2 값 확인 | 3 빈 값 · 4 분석표 | 5 표 1 · 6 비율 · 비교 | 7 해석 · 보고 |

### 0단계 — 데이터 파악하기

분석 전에 어떤 표 · 열이 있고, 한 행이 무엇인지 확인합니다.

**질문** 어느 표의 어느 열에 질환 · 등록 인원 · 상태가 들어 있나?

결과가 보이게 `SELECT * FROM ( )` 로 감쌉니다.

In [ ]:
%%sql
-- [4번] 표 14개: studies, browse_conditions …
SELECT * FROM (SHOW TABLES);

In [ ]:
%%sql
-- [5번] 열 이름과 타입을 보여 줌
SELECT * FROM (DESCRIBE studies);

In [ ]:
%%sql
-- [6번] 실제 값의 모양 확인 (예: 'PHASE3')
SELECT * FROM studies LIMIT 5;

**DESCRIBE studies 결과 (일부)**

| column_name | column_type | 뜻 |
|---|---|---|
| nct_id | VARCHAR | 시험 번호 (키) |
| phase | VARCHAR | 임상 단계 |
| overall_status | VARCHAR | 진행 상태 |
| enrollment | BIGINT | 등록 인원 |
| results_first_posted_date | DATE | 결과 게시일 |

**알게 된 것** 질환은 `browse_conditions.mesh_term`, 나머지는 `studies` 에 있고 `nct_id` 로 연결됩니다.

**한 행** `studies` 는 시험 1건, `browse_conditions` 는 시험 × 질환 1쌍입니다.

### 1단계 — 분석 대상 고르기

질문에 맞는 대상이 몇 건인지 먼저 셉니다.

**질문** 제2형 당뇨병을 대상으로 한 임상시험은 몇 건인가?

**확인** 행수 ≠ 시험수이면 한 시험이 같은 질환으로 두 번 이상 붙은 것입니다. 그래서 `COUNT(DISTINCT nct_id)` 로 셉니다.

**대상** 제2형 당뇨병 시험 1,353건. 이 숫자가 이후 모든 단계의 출발점입니다.

In [ ]:
%%sql
-- [7번] 1단계 → 행수 1,353 / 시험수 1,353
SELECT COUNT(*) AS 행수,
       COUNT(DISTINCT nct_id) AS 시험수
FROM browse_conditions
WHERE mesh_term = 'Diabetes Mellitus, Type 2';

### 2단계 — 값 살펴보기

쓸 열마다 빈 값과 값의 종류를 확인합니다.

**질문** 대상 1,353건에서 등록 인원 · 단계 · 상태 열은 믿을 만한가?

**찾은 것** 등록 인원은 빈 값이 없습니다. 단계는 빈 값이 242건이고, 글자 'NA' 도 값으로 들어 있습니다.

**DuckDB** `SUMMARIZE studies;` 한 줄로 열마다 빈 값 비율 · 최소 · 최대를 볼 수도 있습니다.

In [ ]:
%%sql
-- [8번] 2단계 → 1,353 / 1,353 / 1,111 / 8 / 9
SELECT COUNT(*) AS 대상,
       COUNT(enrollment) AS 인원_있음,
       COUNT(phase) AS 단계_있음,
       COUNT(DISTINCT phase) AS 단계_종류,
       COUNT(DISTINCT overall_status) AS 상태_종류
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2');

### 3단계 — 빈 값 거르기

2단계에서 찾은 빈 값을 분석에서 빼고, 뺀 건수를 기록합니다.

**질문** 등록 인원이 비어 있는 시험을 빼면 몇 건이 남나?

**왜 빼나** 빈 값을 0으로 채우면 평균이 실제보다 낮아집니다. 빼는 것이 원칙입니다.

**기록** 뺀 건수를 보고서에 적습니다. 분석에서는 `WHERE enrollment IS NOT NULL` 을 붙입니다.

**이 자료에서는** 등록 인원 빈 값 0건. 단계(phase) 빈 값 242건은 5단계 표에서 None 행으로 보입니다.

In [ ]:
%%sql
-- [9번] 3단계 → 1,353 / 0 / 1,353
SELECT COUNT(*) AS 대상,
       COUNT(*) - COUNT(enrollment) AS 뺀_건수,
       COUNT(enrollment) AS 남은_건수
FROM studies
WHERE nct_id IN (SELECT nct_id FROM browse_conditions
                 WHERE mesh_term = 'Diabetes Mellitus, Type 2');

### 4단계 — 중복 정리하고 분석표 만들기

뒤의 분석에 쓸 열만 골라, 한 행 = 시험 1건인 표로 저장합니다.

**질문** 빈 값을 뺀 당뇨병 시험을 분석표 하나로 만들려면?

**확인** 행 수 = 시험 수이면 한 행 = 시험 1건입니다.

**왜 JOIN** 다른 표의 열까지 가져올 때는 JOIN을 쓰고 DISTINCT로 중복을 정리합니다.

In [ ]:
%%sql
-- [10번] 4단계 → 분석표 만들기 (OR REPLACE: 다시 실행해도 됨)
CREATE OR REPLACE TEMP TABLE 분석표 AS
SELECT DISTINCT studies.nct_id, phase, enrollment,
       overall_status, results_first_posted_date
FROM studies
JOIN browse_conditions
  ON studies.nct_id = browse_conditions.nct_id
WHERE mesh_term = 'Diabetes Mellitus, Type 2'
  AND enrollment IS NOT NULL;

In [ ]:
%%sql
-- [11번] 확인 → 1,353 / 1,353
SELECT COUNT(*) AS 행수,
       COUNT(DISTINCT nct_id) AS 시험수
FROM 분석표;

### 5단계 — 기술통계표(표 1) 만들기

분석표 하나로 논문 ‘표 1’의 기술통계를 한 번에 냅니다.

**질문** 임상 단계별로 시험 수와 등록 인원의 평균(SD) · 중앙값(IQR)은?

**확인** 평균이 중앙값보다 훨씬 크면 오른쪽으로 치우친 분포 → 중앙값(Q1–Q3)으로 보고합니다.

In [ ]:
%%sql
-- [12번] 5단계 → 9행 (PHASE3 중앙값 321, PHASE1 32)
SELECT phase AS 단계,
       COUNT(*) AS 시험수,
       ROUND(AVG(enrollment)) AS 평균,
       ROUND(STDDEV(enrollment)) AS 표준편차,
       MEDIAN(enrollment) AS 중앙값,
       QUANTILE_CONT(enrollment, 0.25) AS Q1,
       QUANTILE_CONT(enrollment, 0.75) AS Q3
FROM 분석표
GROUP BY phase
ORDER BY phase;

### 6단계 — 비율 구하고 비교하기

분석표에서 분모와 분자를 정하고, 하위 집단끼리 비교합니다.

**질문** 등록 인원 500명 이상인 대규모 시험의 비율은? 한국 참여 시험은 다른가?

**분모** ‘분석표의 시험’ = 분모, ‘500명 이상’ = 분자. 분모를 먼저 말로 정합니다.

**비교** 차이가 의미 있는지는 통계 검정(카이제곱 등)으로 판단합니다.

In [ ]:
%%sql
-- [13번] 6단계 전체 → 1,353 / 209 / 15.4
SELECT COUNT(*) AS 시험수,
       COUNT(CASE WHEN enrollment >= 500 THEN 1 END) AS 대규모,
       ROUND(100.0 * COUNT(CASE WHEN enrollment >= 500 THEN 1 END)
             / COUNT(*), 1) AS 대규모_비율
FROM 분석표;

In [ ]:
%%sql
-- [14번] 한국 참여만 → 76 / 20 / 26.3
SELECT COUNT(*) AS 시험수,
       COUNT(CASE WHEN enrollment >= 500 THEN 1 END) AS 대규모,
       ROUND(100.0 * COUNT(CASE WHEN enrollment >= 500 THEN 1 END)
             / COUNT(*), 1) AS 대규모_비율
FROM 분석표
WHERE nct_id IN (SELECT nct_id FROM countries
                 WHERE name = 'South Korea' AND removed = false);

### 7단계 — 해석하고 보고하기

숫자와 함께 ‘어떻게 정리했는지’를 적어야 결과를 믿을 수 있습니다.

**표 1** 임상 단계별 등록 인원 (5단계 결과 일부)

| 단계 | 시험 수 | 중앙값 (Q1–Q3) |
|---|---:|---|
| PHASE1 | 115 | 32 (24–50) |
| PHASE2 | 88 | 79.5 (30.75–177.25) |
| PHASE3 | 121 | 321 (166–608) |
| PHASE4 | 147 | 88 (30–192) |
| NA | 611 | 60 (30–133.5) |

**해석** 3상 시험의 등록 인원 중앙값은 321명으로, 1상(32명)의 약 10배입니다.

**보고서에 함께 적을 것**

- 대상: mesh_term = 제2형 당뇨병, 1,353건
- 제외: 등록 인원 빈 값 0건 (3단계)
- 중복: DISTINCT로 한 행 = 시험 1건 (4단계)
- 통계: 중앙값(Q1–Q3) · 대규모 기준 500명 (5 · 6단계)
- 기준: 자료 추출일 2026-09-19

**다음 단계** 분석표를 CSV로 내보내 R · Python · SPSS에서 검정 · 회귀분석을 합니다.

In [ ]:
%%sql
-- [15번] (선택) CSV로 내보내기 → 왼쪽 파일 목록에 분석표.csv
COPY 분석표 TO '분석표.csv' (HEADER);

### 여기까지 정리

① **준비**: 표 · 열을 파악하고, 대상을 고르고, 빈 값 · 값의 종류를 확인한다

② **정리**: 빈 값을 빼고, DISTINCT로 한 행 = 시험 1건인 분석표를 만든다

③ **분석 · 보고**: 분석표에서 표 1 · 비율을 내고, 정리 과정을 함께 적는다

**잠깐 생각해 봅시다** 4단계 분석표의 행 수가 시험 수보다 많다면 무엇을 의심해야 할까요?

→ JOIN 때문에 같은 시험이 여러 행으로 붙은 것입니다. DISTINCT를 빠뜨렸는지 확인합니다.

## 3. 병원 SQL 사례 — 실제로 이렇게 씁니다

### 병원에서 SQL을 쓰는 일 — 실제로 이런 숫자를 SQL로 뽑습니다

| 누가 | 무엇을 | 어떤 SQL이 쓰이나 |
|---|---|---|
| QI · 적정성평가 담당 | 지표별 대상자 수와 충족률 | 조건으로 대상자 선별 → 분모 · 분자 각각 COUNT |
| 감염관리팀 | 월별 감염 발생 건수, 재원일수 | GROUP BY 월 · 병동, 날짜 차이 계산 |
| 간호부 | 간호 인력 대비 환자 수, 낙상 건수 | GROUP BY 병동 · 근무조, COUNT와 비율 |
| 연구자 | 연구 대상자 선별, 변수 추출 | 여러 표를 JOIN하여 코호트 추출 |
| 원무 · 경영 | 진료과별 실적, 수납 현황 | GROUP BY 진료과 · 기간, SUM과 AVG |

공통점: **대상을 고르고(WHERE) → 묶어서 세고(GROUP BY) → 검산한다**. 오늘 배운 그대로입니다.

코호트 = 같은 조건으로 묶어 분석 대상으로 삼은 집단 (13주차에서 자세히).

### 사례 — 30일 재입원율

“지난달 퇴원 환자 중 30일 안에 다시 입원한 비율”을 구한다고 해 봅시다.

| 분모 — 대상 퇴원 | 분자 — 재입원 |
|---|---|
| 지난달에 퇴원한 입원 건 | 그 퇴원일로부터 30일 이내 |
| 사망 · 전원은 제외 | 계획된 재입원은 제외 |
| 같은 환자가 여러 번이면 각각 셈 | 같은 환자의 다음 입원을 찾아 연결 |
| → `COUNT(*) FROM 입원 WHERE …` | → 퇴원 기록과 입원 기록을 JOIN |

어려운 부분은 SQL 문법이 아니라 **제외 기준을 정하는 일**입니다.

13주차에서 기준일(index date)을 정하는 방법과 함께 자세히 다룹니다.

## 4. 검산 — 결과를 믿기 전에

결과를 보고서에 쓰기 전에 **반드시 확인하는 다섯 가지**입니다.

| 확인할 것 | 무엇을 보는가 | 어떻게 |
|---|---|---|
| ① 전체 건수 | 조건 없이 센 값과 비교했을 때 말이 되는가 | `COUNT(*) FROM studies` |
| ② 세는 단위 | 시험 수인가, 행 수인가 | `COUNT(*)` 인가 `COUNT(DISTINCT nct_id)` 인가 |
| ③ 분모 | JOIN 때문에 대상이 줄지 않았는가 | `LEFT JOIN` 으로 다시 세기 |
| ④ 빠진 값 | NULL이 많은 열을 조건에 썼는가 | `COUNT(열)` 로 값 있는 행 세기 |
| ⑤ 이상한 값 | 최소 · 최대가 상식적인가 | `MIN()`, `MAX()` 로 확인 |

In [ ]:
%%sql
-- [16번] ⑤ 이상한 값 → 최소 0명, 최대 4,000,000명. 상식적인가?
SELECT MIN(enrollment) AS 최소인원,
       MAX(enrollment) AS 최대인원
FROM 분석표;

숫자가 틀리는 것보다 **틀린 줄 모르는 것**이 더 위험합니다.

12주차에서 이 다섯 가지를 오류 사례와 함께 다시 봅니다.

## 5. 과제 1 — 제출 안내

20점 · 노트북 파일 하나 · 9주차 수업 후 1주일 이내

**무엇을 제출하나요**

- 노트북 파일 하나 (`과제1.ipynb`)
- 모든 셀을 실행한 상태로 저장
- 학번 · 이름을 첫 셀에 기입
- 마지막 해석 문항은 직접 작성
- 제출: LMS, 마감: 9주차 수업 후 1주일 이내

**어떻게 채점하나요**

- 자동 확인 12점: `check("Q1", _)` 가 ✅ 이면 정답
- 결과 해석 5점: 숫자가 뜻하는 바를 문장으로
- 실행 완료 · 형식 3점

| 문항 유형 | 개수 | 내용 |
|---|---|---|
| ① 조회 기초 | 6문항 | SELECT · WHERE · ORDER BY 의 잘못된 SQL 고치기 (7주차) |
| ② 집계와 GROUP BY | 3문항 | COUNT · AVG · GROUP BY 의 잘못된 SQL 고치기 (8주차) |
| ③ 당뇨병 시험 분석 | 3문항 | JOIN 으로 두 표를 연결해 세기 |
| ④ 결과 해석 | 2문항 | 숫자가 뜻하는 바를 문장으로 |

**푸는 법**

- SQL마다 한 군데가 틀려 있습니다. 고쳐서 ✅ 를 받습니다.
- 무엇이 왜 틀렸는지 한 줄로 적습니다.

## 직접 해보기 — 위암으로 연습

질환만 위암(`'Stomach Neoplasms'`)으로 바꿉니다.

빈칸 `___` 을 채워 실행한 뒤 아래 check 셀을 실행하세요.

**Q1. 서브쿼리** — 위암 대상 시험이 몇 건인지 서브쿼리로 세세요.

In [ ]:
%%sql
-- [17번] Q1
SELECT COUNT(DISTINCT nct_id) AS 시험수
FROM studies
WHERE nct_id ___ (
    SELECT ___
    FROM browse_conditions
    WHERE mesh_term = 'Stomach Neoplasms'
);

In [ ]:
check("Q1", _)   # [18번]

**Q2. 2단계 값 살펴보기** — 위암 시험 중 단계(`phase`)에 값이 있는 시험 수(`단계_있음`)를 세세요.

In [ ]:
%%sql
-- [19번] Q2
SELECT COUNT(*) AS 대상,
       COUNT(___) AS 단계_있음
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Stomach Neoplasms'
);

In [ ]:
check("Q2", _)   # [20번]

**Q3. 6단계 비율** — 위암 시험 중 등록 인원 500명 이상(대규모)의 수와 비율을 구하세요. 두 `___` 에 같은 조건을 적습니다.

In [ ]:
%%sql
-- [21번] Q3
SELECT COUNT(*) AS 시험수,
       COUNT(CASE WHEN ___ THEN 1 END) AS 대규모,
       ROUND(100.0 * COUNT(CASE WHEN ___ THEN 1 END)
             / COUNT(*), 1) AS 대규모_비율
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Stomach Neoplasms'
);

In [ ]:
check("Q3", _)   # [22번]

**Q4. 6단계 비교** — Q3 에 한국 참여 조건을 더했습니다. 나라 이름을 채우세요.

In [ ]:
%%sql
-- [23번] Q4
SELECT COUNT(*) AS 시험수,
       COUNT(CASE WHEN enrollment >= 500 THEN 1 END) AS 대규모,
       ROUND(100.0 * COUNT(CASE WHEN enrollment >= 500 THEN 1 END)
             / COUNT(*), 1) AS 대규모_비율
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Stomach Neoplasms'
)
  AND nct_id IN (SELECT nct_id FROM countries
                 WHERE name = ___ AND removed = false);

In [ ]:
check("Q4", _)   # [24번]

**Q5. 조건을 하나씩 더하기** — 아래 예제(위암 + 중재 연구)를 실행한 뒤, Q5 셀에 완료 조건 한 줄을 AND 로 더하세요 (완료 = 'COMPLETED').

In [ ]:
%%sql
-- [25번] 예제: 위암 + 중재 연구 → 151
SELECT COUNT(*) AS 시험수
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Stomach Neoplasms'
)
  AND study_type = 'INTERVENTIONAL';

In [ ]:
%%sql
-- [26번] Q5
SELECT COUNT(*) AS 시험수
FROM studies
WHERE nct_id IN (
    SELECT nct_id
    FROM browse_conditions
    WHERE mesh_term = 'Stomach Neoplasms'
)
  AND study_type = 'INTERVENTIONAL'
  ___

In [ ]:
check("Q5", _)   # [27번]

**Q6 (선택). 검산 ⑤ 이상한 값** — 위암 시험의 등록 인원 최소 · 최대를 구하세요. 괄호 안을 직접 적습니다.

In [ ]:
%%sql
-- [28번] Q6
SELECT MIN(enrollment) AS 최소인원,
       MAX(enrollment) AS 최대인원
FROM studies
WHERE nct_id IN (
    ___
);

In [ ]:
check("Q6", _)   # [29번]

## 6. 정리

### 9주차 용어 정리 — 이번 시간에 새로 나온 말

| 용어 | 영어 | 뜻 (한 줄) |
|---|---|---|
| 세는 단위 | Granularity | 결과의 한 행이 무엇인지 (환자 · 방문 · 시험) |
| 질문의 네 요소 | — | 대상 · 기간 · 조건 · 세는 단위. SQL로 옮기기 전에 적는다 |
| 단계별 추출 | — | 조건을 하나씩 더하며 건수 변화를 기록하는 방법 |
| 서브쿼리 | Subquery | 괄호 안이 먼저 계산되는 SQL. 대상을 고를 때 씀 |
| 검산 | Sanity Check | 숫자를 내보내기 전에 스스로 확인하는 절차 |
| 분모 · 분자 | Denominator · Numerator | 비율을 만들 때 아래와 위에 오는 수 |
| 이상값 | Outlier | 혼자 크게 튀어 평균을 끌어당기는 값 |

## 오늘 배운 것

- **준비**: 표 · 열을 파악하고, 대상을 고르고, 빈 값 · 값의 종류를 확인한다
- **정리**: 빈 값을 빼고, DISTINCT로 한 행 = 시험 1건인 분석표를 만든다
- **분석**: 분석표에서 표 1(평균 · 중앙값)과 비율을 낸다
- **보고**: 빼고 정리한 과정을 숫자와 함께 적고, 검산 뒤에 내보낸다

다음 시간: 10강. 안전한 데이터 수정과 트랜잭션 — INSERT · UPDATE · DELETE

아래 셀로 오늘 문항을 확인합니다.

In [ ]:
진행확인()   # [30번]